# BEM layers from CHARM parcellation

## 🧪 - Experiment setup
- `seg`: CHARM head segmentation

In [1]:
# imports
import ants
import mne
import mindmap as mm

# readers
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# data
brain = dm.load(sub, "brain.mgz")
charm = dm.load(sub, "seg")
ras2tkras = dm.load(sub, "ras2tkras")    # to build meshes in FreeSurfer space (expected by MNE)
vol_info = dm.load(sub, "vol-info")      # metadata for writing surfaces
bem_labels = dm.get_bem_labels()


## 🧬 - Data Processing
Build Boundary Element Model

In [2]:
# brain surface
brain_mask = mm.anatomy.mri.nibabel_to_ants(brain)
brain_mask = ants.threshold_image(brain_mask, low_thresh=0.5, high_thresh=None, inval=1, outval=0)
brain_mask = brain_mask.morphology("close", radius=10)

brain_surf, brain_interop = mm.anatomy.forward.fit_surface_to_mri_mask(brain_mask, n_smooth=20)
brain_surf = mm.geometry.polydata_transform(brain_surf, ras2tkras)      # CHECK IF THIS LINE FITS THE MESH

# build BEM layers from charm segmentation
layers = mm.anatomy.forward.build_bem_layers(charm, bem_labels, brain_interp=brain_interop, n_smooth=20)
layers = mm.anatomy.forward.bem_transform(layers, ras2tkras)

# save layers in freesurfer/{subject}/bem
folder = dm.roots["fs"]/f"sub-{sub}"/"bem"
folder.mkdir(exist_ok=True)
mm.anatomy.forward.save_bem_as_freesurfer(folder, layers, vol_info, overwrite=True)

# assemble
model = mne.make_bem_model(
    subject      = f"sub-{sub}",
    subjects_dir = dm.roots["fs"],
    conductivity = [0.3, 0.006, 0.3],    # conductivities (brain, skull, skin)
    ico          = None,                 # mesh resolution (simplifies computations), tris don't match though
)

bem  = mne.make_bem_solution(model)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Creating the BEM geometry...
outer skin  CM is  -1.27   5.47  -8.03 mm
outer skull CM is  -0.83   4.76   8.69 mm
inner skull CM is  -0.11  -6.49  24.90 mm
Checking that surface outer skull is inside surface outer skin  ...
Checking that surface inner skull is inside surface outer skull ...
Checking distance between outer skin  and outer skull surfaces...
Minimum distance between the outer skin  and outer skull surfaces is approximately    3.9 mm
Checking distance between outer skull and inner skull surfaces...
Minimum distance between the outer skull and inner skull surfaces is approximately    0.3 mm
Surfaces passed the basic topology checks.
Complete.

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (10242) -> outer skin  (10242) ...


/tmp/ipykernel_7856/2524034581.py:19: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  model = mne.make_bem_model(
/tmp/ipykernel_7856/2524034581.py:26: RuntimeWarning: The bem surfaces have 10242 data points. 5120 (ico grade=4) should be enough. Dense 3-layer bems may not save properly.
  bem  = mne.make_bem_solution(model)


        outer skin  (10242) -> outer skull (10242) ...
        outer skin  (10242) -> inner skull (10242) ...
        outer skull (10242) -> outer skin  (10242) ...
        outer skull (10242) -> outer skull (10242) ...
        outer skull (10242) -> inner skull (10242) ...
        inner skull (10242) -> outer skin  (10242) ...
        inner skull (10242) -> outer skull (10242) ...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (10242) -> inner skull (10242) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


## 📊 - Visualization

Visualize in freeview

In [5]:
mm.viz.plot_bem(dm.roots["fs"], subject="simf10")

Visualize 3D

In [ ]:
import pyvista as pv
p = pv.Plotter()
#p.add_mesh(brain_surf, style="wireframe", color="pink", line_width=1)
p.add_mesh(layers["skin"], style="wireframe", color="green", line_width=1)
p.add_mesh(layers["outer skull"], style="wireframe", color="black", line_width=1)
p.add_mesh(layers["inner skull"], style="wireframe", color="red", line_width=1)
p.show()

Widget(value='<iframe src="http://localhost:34925/index.html?ui=P_0x7024f1761010_1&reconnect=auto" class="pyvi…